# Project Phase 2: Feature Engineering (特徴量候補の可視化と検討)

このノートブックでは、BOJ金利のスプレッド（プレミアム）と、外部指標の移動平均乖離率 vs 分数階差を可視化し、どちらがモデルにとって有益なシグナルかを検討します。

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.features import generate_features

%matplotlib inline
sns.set(style='whitegrid')
plt.rcParams['font.family'] = 'AppleGothic'

## 1. 特徴量の生成実行 (Spread, MA_Dev, FracDiff)

In [ ]:
df_clean = pd.read_csv('../data/cleaned_data.csv')
df_clean['日付'] = pd.to_datetime(df_clean['日付'])

df_featured = generate_features(df_clean, ma_window=20, d=0.4)
df_featured.to_csv('../data/featured_data.csv', index=False)
print(f'Feature generated. Data shape: {df_featured.shape}')

## 2. BOJ Swap 特徴量の比較 (Spread vs FracDiff)
M1-M8のスプレッド（利上げプレミアム）を一括でプロットします。

In [ ]:
plt.figure(figsize=(15, 8))
boj_cols = ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8']
colors = plt.cm.plasma(np.linspace(0, 1, len(boj_cols)))

for i, m in enumerate(boj_cols):
    plt.plot(df_featured['日付'], df_featured[f'{m}_spread'], label=f'{m} spread', color=colors[i], alpha=0.7)

plt.title('All BOJ OIS Spreads (M(n) - Actual Policy Rate)', fontsize=16)
plt.ylabel('Spread (%)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

## 3. 外部指標 特徴量の比較 (MA20 Dev vs FracDiff)
USDJPY と Nikkei225 について、移動平均乖離率と分数階差を並べて比較します。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12), sharex=True)

# USDJPY
axes[0, 0].plot(df_featured['日付'], df_featured['USDJPY_ma_dev'], color='blue')
axes[0, 0].set_title('USDJPY: MA20 Deviation Ratio')

axes[0, 1].plot(df_featured['日付'], df_featured['USDJPY_frac_diff'], color='darkblue')
axes[0, 1].set_title('USDJPY: Fractional Diff (d=0.4)')

# Nikkei225
axes[1, 0].plot(df_featured['日付'], df_featured['Nikkei225_ma_dev'], color='red')
axes[1, 0].set_title('Nikkei225: MA20 Deviation Ratio')

axes[1, 1].plot(df_featured['日付'], df_featured['Nikkei225_frac_diff'], color='darkred')
axes[1, 1].set_title('Nikkei225: Fractional Diff (d=0.4)')

plt.tight_layout()
plt.show()